# Laboratório guiado: Lakehouse financeiro e DRE no Databricks Free Edition

**Objetivo:** construir um pipeline Bronze → Silver → Gold, usando **Python/PySpark para ingestão** e **SQL para transformação e indicadores financeiros**. Ao final, você terá uma DRE mensal por empresa, margens e crescimento de receita.

**Público:** iniciantes em Databricks, Engenharia de Dados e Analytics Financeiro. **Duração sugerida:** 90–150 minutos.

> Este laboratório usa dados sintéticos e uma DRE **gerencial simplificada**, não um demonstrativo contábil auditado. Não requer funcionalidades Enterprise, armazenamento externo, Auto Loader ou Jobs. A disponibilidade e as cotas da Free Edition podem mudar.

## Roteiro de aprendizagem

| Etapa | Linguagem | Conceito do Databricks | Entregável |
|---|---|---|---|
| Ambiente | SQL | Workspace, notebook, compute serverless e catálogo | Identificar contexto de execução |
| Bronze | Python/PySpark | DataFrames, execução distribuída, tabelas Delta | Dados brutos persistidos |
| Silver | SQL | `TRY_CAST`, janela `ROW_NUMBER`, validações | Lançamentos padronizados |
| Gold | SQL | Views, agregações, CTEs, indicadores | DRE mensal |
| Análise | SQL | Consultas, visualização e verificação | Métricas e testes |

**Modelo de dados:** cada linha representa um lançamento financeiro com empresa, competência, categoria contábil, conta e valor. Valores são positivos; a regra de sinal é aplicada na Gold.

## 1. Conhecendo o ambiente Databricks

O **workspace** é onde ficam os notebooks. Cada notebook combina células Markdown (explicação) e código. O **compute** executa Spark e SQL; no Free Edition, use o recurso serverless que aparecer no seu ambiente. O **catálogo** organiza objetos em `catalog.schema.table`. Neste laboratório, `default.tabela` significa uma tabela no schema `default` do catálogo atual.

**Antes de começar:** importe este `.ipynb` no workspace, abra o notebook e conecte-o ao compute disponível. As células SQL começam com `%sql`, que instrui o Databricks a executar a célula como SQL. Não rode `%sql` em Jupyter Python convencional.

**Verificação:** a próxima consulta mostra o catálogo e schema atuais. Se você não tiver permissão de escrita no schema `default`, substitua `default` por um schema no qual possa criar tabelas, em todas as células.

In [0]:
%sql
SELECT current_catalog() AS catalogo, current_schema() AS schema_atual;

catalogo,schema_atual
workspace,default


### O que observar na saída?

`current_catalog()` indica o catálogo selecionado e `current_schema()` o schema atual. O nome totalmente qualificado evita ambiguidades entre tabelas homônimas. **Atenção:** o laboratório cria ou substitui objetos chamados `bronze_dre_lancamentos`, `silver_dre_lancamentos`, `silver_dre_rejeitados` e `gold_dre_mensal`; execute em um schema de estudo, não de produção.

## 2. Camada Bronze: dados de origem sem regras de negócio

Na **Medallion Architecture**, Bronze prioriza rastreabilidade e preservação do que chegou da fonte. Aqui simulamos um ERP, pois a Free Edition pode não oferecer acesso a uma origem externa. O gerador usa `spark.range`, evitando criar um grande conjunto de linhas na memória do computador local.

**Conceitos:**
- **Spark DataFrame:** coleção distribuída de linhas e colunas; transformações como `withColumn` descrevem um plano de execução.
- **Lazy evaluation:** as transformações são executadas quando uma ação, como `count`, é solicitada.
- **Lote incremental simulado:** `LOTE_ID` representa uma entrega do ERP. Mudar o número gera novas chaves de lançamento.
- **Coluna `ingestao_em`:** metadado de quando o dado entrou no lakehouse, útil para auditoria.

**Experimento:** execute a célula e observe o total preparado. Não é necessário instalar bibliotecas adicionais.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

LOTE_ID = 2            # altere para 2, 3... para simular novas ingestões
QTD_LANCAMENTOS = 40000  # aumente com cuidado conforme limites da Free Edition
TABELA_BRONZE = "default.bronze_dre_lancamentos"

contas = [
 ("RECEITA_BRUTA", "VENDAS", 44),
 ("DEDUCOES", "IMPOSTOS_VENDAS", 8),
 ("DEDUCOES", "DEVOLUCOES", 3),
 ("CMV", "CUSTO_VENDAS", 22),
 ("DESPESA_OPERACIONAL", "PESSOAL", 10),
 ("DESPESA_OPERACIONAL", "MARKETING", 4),
 ("DESPESA_OPERACIONAL", "ADMINISTRATIVAS", 4),
 ("DEPRECIACAO", "DEPRECIACAO", 2),
 ("RESULTADO_FINANCEIRO", "RECEITA_FINANCEIRA", 1),
 ("RESULTADO_FINANCEIRO", "DESPESA_FINANCEIRA", 1),
 ("IR_CSLL", "IR_CSLL", 1)
]
# Distribuição por faixa de percentuais cumulativos, determinística por ID/lote
intervalos=[]
acum=0
for categoria, conta, peso in contas:
    intervalos.append((acum, acum+peso, categoria, conta))
    acum+=peso
assert acum==100

df = (spark.range(QTD_LANCAMENTOS)
 .withColumn("lote_id", F.lit(LOTE_ID))
 .withColumn("id_lancamento", F.concat_ws("-",F.lit("ERP"),F.col("lote_id"),F.col("id")))
 .withColumn("empresa_id", F.when(F.col("id")%3==0,"EMP_A").otherwise("EMP_B"))
 .withColumn("data_competencia",F.date_format(F.date_add(F.lit("2025-01-01").cast("date"),((F.col("id")*17+LOTE_ID*13)%540).cast("int")),"yyyy-MM-dd"))
 .withColumn("faixa",((F.col("id")*37+LOTE_ID*11)%100).cast("int"))
 .withColumn("valor",F.round(F.lit(100.0)+((F.col("id")*193+LOTE_ID*71)%15000).cast("double"),2))
 .withColumn("moeda",F.lit("BRL"))
 .withColumn("origem",F.lit("ERP_SIMULADO")))
expr_cat = F.lit(None).cast("string")
expr_conta = F.lit(None).cast("string")
for inicio, fim, cat, conta in reversed(intervalos):
    cond=(F.col("faixa")>=inicio)&(F.col("faixa")<fim)
    expr_cat=F.when(cond,F.lit(cat)).otherwise(expr_cat)
    expr_conta=F.when(cond,F.lit(conta)).otherwise(expr_conta)
df=(df.withColumn("categoria",expr_cat).withColumn("conta",expr_conta)
    .withColumn("ingestao_em",F.current_timestamp())
    .select("id_lancamento","lote_id","empresa_id","data_competencia","categoria","conta","valor","moeda","origem","ingestao_em"))
df.createOrReplaceTempView("novos_lancamentos_bronze")
print("Lote preparado:", LOTE_ID, "registros:",df.count())

Lote preparado: 2 registros: 40000


### Interpretando os campos gerados

| Campo | Significado | Uso posterior |
|---|---|---|
| `id_lancamento` | Identificador único na origem | Chave de deduplicação |
| `lote_id` | Identificador da remessa | Simulação incremental |
| `empresa_id` | Unidade analisada | Agrupamento da DRE |
| `data_competencia` | Data contábil da operação | Consolidação mensal |
| `categoria` e `conta` | Classificação gerencial | Linhas da DRE |
| `valor` | Montante do lançamento | Soma e margens |
| `ingestao_em` | Momento da ingestão | Rastreabilidade |

A distribuição de contas é didática e **não garante margens economicamente realistas**. O objetivo é exercitar transformações e regras de cálculo.

## 3. Persistência Bronze: por que Delta Lake?

**Delta Lake** adiciona transações ACID e suporte a operações como `MERGE` sobre tabelas do lakehouse. Diferentemente de apenas salvar um arquivo, uma tabela Delta permite atualizações controladas e consultas SQL consistentes.

Nesta célula, Python usa `spark.sql(...)` para criar a tabela e executar um **upsert** (`MERGE`): quando o ID já existe, atualiza; caso contrário, insere. Reexecutar o mesmo lote não aumenta a quantidade de IDs. Esse comportamento é chamado de **idempotência**.

**Observação:** como `ingestao_em` é recalculado, reprocessar um lote altera o timestamp de ingestão do registro correspondente. Em sistemas reais, pode ser desejável manter também a primeira data de chegada.

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {TABELA_BRONZE} (
 id_lancamento STRING, lote_id BIGINT, empresa_id STRING, data_competencia STRING,
 categoria STRING, conta STRING, valor DOUBLE, moeda STRING, origem STRING,
 ingestao_em TIMESTAMP) USING DELTA""")
spark.sql(f"""MERGE INTO {TABELA_BRONZE} AS destino
USING novos_lancamentos_bronze AS origem
ON destino.id_lancamento = origem.id_lancamento
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *""")
print("Bronze total:",spark.table(TABELA_BRONZE).count())

Bronze total: 60000


### Verificando a Bronze com SQL

O Databricks permite alternar Python e SQL no mesmo notebook. A consulta a seguir confirma o volume por lote. Você também pode abrir a tabela no explorador de catálogo do workspace.

**Prática:** execute a consulta abaixo e compare `quantidade` com `QTD_LANCAMENTOS`. Depois de gerar o lote 2, deverão aparecer dois grupos.

In [0]:
%sql
SELECT lote_id, COUNT(*) AS quantidade, MIN(ingestao_em) AS primeira_ingestao, MAX(ingestao_em) AS ultima_ingestao
FROM default.bronze_dre_lancamentos
GROUP BY lote_id ORDER BY lote_id;

lote_id,quantidade,primeira_ingestao,ultima_ingestao
1,20000,2026-10-11T05:49:06.917Z,2026-10-11T05:49:06.917Z
2,40000,2026-10-11T05:52:29.706Z,2026-10-11T05:52:29.706Z


## 4. Camada Silver: qualidade, tipos e deduplicação

A **Silver** transforma dados brutos em uma base confiável para análises. Aqui aplicamos SQL declarativo, isto é, descrevemos **o resultado desejado**, e o mecanismo SQL do Databricks planeja a execução.

**O que a transformação faz:** `TRY_CAST` tenta converter datas e valores sem interromper a consulta em caso de formato inválido; `TRIM` remove espaços e `UPPER` normaliza a moeda; `ROW_NUMBER()` atribui uma posição por ID para escolher a versão mais recente; os filtros mantêm apenas registros válidos em BRL.

**Por que `CREATE OR REPLACE TABLE`?** Facilita a aula e a reexecução, mas recalcula todos os dados. Para volumes altos, considere ingestão incremental com `MERGE` e controle de mudanças.

**Atenção à deduplicação:** a janela prioriza `ingestao_em` e depois `lote_id`. Se duas linhas tiverem a mesma chave e timestamps idênticos, é necessário um critério adicional de desempate para garantir determinismo.

In [0]:
%sql
CREATE OR REPLACE TABLE default.silver_dre_lancamentos USING DELTA AS
WITH tipado AS (
 SELECT id_lancamento, CAST(lote_id AS BIGINT) lote_id, empresa_id,
        TRY_CAST(data_competencia AS DATE) data_competencia,
        categoria, conta, TRY_CAST(valor AS DECIMAL(18,2)) valor,
        UPPER(TRIM(moeda)) moeda, origem, ingestao_em,
        ROW_NUMBER() OVER (PARTITION BY id_lancamento ORDER BY ingestao_em DESC, lote_id DESC) rn
 FROM default.bronze_dre_lancamentos
)
SELECT id_lancamento, lote_id, empresa_id, data_competencia, categoria,
       conta, valor, moeda, origem, ingestao_em
FROM tipado
WHERE rn=1 AND id_lancamento IS NOT NULL AND empresa_id IS NOT NULL
  AND data_competencia IS NOT NULL AND valor IS NOT NULL AND valor >= 0
  AND moeda='BRL' AND categoria IN
 ('RECEITA_BRUTA','DEDUCOES','CMV','DESPESA_OPERACIONAL',
  'DEPRECIACAO','RESULTADO_FINANCEIRO','IR_CSLL');

num_affected_rows,num_inserted_rows


### Registros rejeitados e observabilidade

Um pipeline confiável não deve apenas descartar dados inválidos sem explicação. A view de rejeitados identifica registros que não atendem às regras de tipo, moeda, categoria e preenchimento. Ela **não inclui duplicatas válidas descartadas pela deduplicação**, que são um problema de qualidade distinto.

**Diferença entre tabela e view:** a tabela Silver materializa dados; a **view** armazena a definição de uma consulta e lê a Bronze no momento da execução. Views são úteis para lógica reutilizável, mas dependem das tabelas de origem.

In [0]:
%sql
CREATE OR REPLACE VIEW default.silver_dre_rejeitados AS
SELECT b.* FROM default.bronze_dre_lancamentos b
WHERE b.id_lancamento IS NULL OR b.empresa_id IS NULL
   OR TRY_CAST(b.data_competencia AS DATE) IS NULL
   OR TRY_CAST(b.valor AS DECIMAL(18,2)) IS NULL
   OR TRY_CAST(b.valor AS DECIMAL(18,2)) < 0
   OR UPPER(TRIM(b.moeda)) <> 'BRL' OR b.moeda IS NULL
   OR b.categoria NOT IN ('RECEITA_BRUTA','DEDUCOES','CMV','DESPESA_OPERACIONAL',
                          'DEPRECIACAO','RESULTADO_FINANCEIRO','IR_CSLL')
   OR b.categoria IS NULL;
SELECT COUNT(*) AS registros_rejeitados FROM default.silver_dre_rejeitados;

registros_rejeitados
0


### Experimento de qualidade (opcional)

Para estudar validações sem modificar dados, rode esta consulta de diagnóstico. Ela revela quantos lançamentos existem em cada categoria na Silver. Se quiser simular rejeições, adapte o gerador de um lote novo para incluir uma moeda diferente de BRL e observe a contagem da view de rejeitados.

In [0]:
%sql
SELECT categoria, COUNT(*) AS lancamentos, ROUND(SUM(valor), 2) AS valor_total
FROM default.silver_dre_lancamentos
GROUP BY categoria ORDER BY categoria;

categoria,lancamentos,valor_total
CMV,13200,100238400.00
DEDUCOES,6600,50119600.00
DEPRECIACAO,1200,9113400.00
DESPESA_OPERACIONAL,10800,82033600.00
IR_CSLL,600,4518600.00
RECEITA_BRUTA,26400,200589400.00
RESULTADO_FINANCEIRO,1200,9087000.00


## 5. Camada Gold: métricas de negócio e DRE

A **Gold** disponibiliza indicadores prontos para consumo por analistas, dashboards e equipes de negócio. Usamos uma **view SQL** mensal, agrupada por empresa e competência.

**Estrutura simplificada da DRE:**

1. Receita bruta − deduções = **receita líquida**.
2. Receita líquida − custo das mercadorias vendidas (CMV) = **lucro bruto**.
3. Lucro bruto − despesas operacionais = **EBITDA** (simplificação didática).
4. EBITDA − depreciação = **EBIT**.
5. EBIT + receitas financeiras − despesas financeiras − IR/CSLL = **lucro líquido**.

`WITH` define **CTEs**, subconsultas nomeadas que tornam o cálculo legível e auditável. `CASE WHEN` converte categorias de lançamentos em colunas financeiras. `NULLIF` evita divisão por zero no cálculo de margens.

**Limitação contábil:** na prática, EBITDA, EBIT, tributos e apresentação da DRE dependem de políticas contábeis, plano de contas, competência e ajustes. O modelo aqui não inclui todos os elementos de uma DRE societária.

In [0]:
%sql
CREATE OR REPLACE VIEW default.gold_dre_mensal AS
WITH agrupado AS (
 SELECT empresa_id, DATE_TRUNC('MONTH',data_competencia) mes,
 SUM(CASE WHEN categoria='RECEITA_BRUTA' THEN valor ELSE 0 END) receita_bruta,
 SUM(CASE WHEN categoria='DEDUCOES' THEN valor ELSE 0 END) deducoes,
 SUM(CASE WHEN categoria='CMV' THEN valor ELSE 0 END) cmv,
 SUM(CASE WHEN categoria='DESPESA_OPERACIONAL' THEN valor ELSE 0 END) despesas_operacionais,
 SUM(CASE WHEN categoria='DEPRECIACAO' THEN valor ELSE 0 END) depreciacao,
 SUM(CASE WHEN conta='RECEITA_FINANCEIRA' THEN valor WHEN conta='DESPESA_FINANCEIRA' THEN -valor ELSE 0 END) resultado_financeiro,
 SUM(CASE WHEN categoria='IR_CSLL' THEN valor ELSE 0 END) ir_csll
 FROM default.silver_dre_lancamentos
 GROUP BY empresa_id, DATE_TRUNC('MONTH',data_competencia)
), calc1 AS (
 SELECT *, receita_bruta-deducoes AS receita_liquida,
 (receita_bruta-deducoes)-cmv AS lucro_bruto
 FROM agrupado
), calc2 AS (
 SELECT *, lucro_bruto-despesas_operacionais AS ebitda FROM calc1
), calc3 AS (
 SELECT *, ebitda-depreciacao AS ebit FROM calc2
)
SELECT *, ebit+resultado_financeiro-ir_csll AS lucro_liquido,
 ROUND(100*lucro_bruto/NULLIF(receita_liquida,0),2) AS margem_bruta_pct,
 ROUND(100*ebitda/NULLIF(receita_liquida,0),2) AS margem_ebitda_pct,
 ROUND(100*(ebit+resultado_financeiro-ir_csll)/NULLIF(receita_liquida,0),2) AS margem_liquida_pct
FROM calc3;

### Como ler o resultado Gold

Margens são calculadas sobre a **receita líquida**. Por exemplo, margem EBITDA = `100 × EBITDA / receita_liquida`. Uma margem negativa sinaliza resultado operacional negativo sob as regras adotadas; uma margem muito alta ou muito baixa pode indicar características dos dados sintéticos, não necessariamente desempenho real.

A view Gold não armazena uma cópia independente: quando a Silver é reconstruída, uma nova consulta à Gold reflete os dados atualizados.

## 6. Análise temporal: crescimento mês contra mês

A função de janela **`LAG`** recupera a receita líquida do mês anterior **disponível** para a mesma empresa. O crescimento MoM é `(receita_atual / receita_anterior − 1) × 100`. No primeiro mês, o resultado é nulo. Se houver meses sem lançamentos, o cálculo compara com o último mês existente, e não necessariamente com o mês calendário imediatamente anterior.

In [0]:
%sql
SELECT empresa_id, DATE_FORMAT(mes,'yyyy-MM') competencia,
 receita_bruta, receita_liquida, lucro_bruto, ebitda, ebit, lucro_liquido,
 margem_bruta_pct, margem_ebitda_pct, margem_liquida_pct,
 ROUND(100*(receita_liquida/NULLIF(LAG(receita_liquida) OVER
 (PARTITION BY empresa_id ORDER BY mes),0)-1),2) crescimento_receita_mom_pct
FROM default.gold_dre_mensal
ORDER BY empresa_id, competencia;

empresa_id,competencia,receita_bruta,receita_liquida,lucro_bruto,ebitda,ebit,lucro_liquido,margem_bruta_pct,margem_ebitda_pct,margem_liquida_pct,crescimento_receita_mom_pct
EMP_A,2025-01,3830141.00,2994758.00,1227116.00,-307955.00,-431375.00,-654536.00,40.98,-10.28,-21.86,null
EMP_A,2025-02,3408885.00,2379428.00,691012.00,-763175.00,-872903.00,-829760.00,29.04,-32.07,-34.87,-20.55
EMP_A,2025-03,4057808.00,3202516.00,1153275.00,-411923.00,-625928.00,-844686.00,36.01,-12.86,-26.38,34.59
EMP_A,2025-04,3612781.00,2579089.00,666473.00,-788889.00,-1011799.00,-957142.00,25.84,-30.59,-37.11,-19.47
EMP_A,2025-05,3817977.00,2981054.00,1173916.00,-411451.00,-525871.00,-746540.00,39.38,-13.80,-25.04,15.59
EMP_A,2025-06,3622196.00,2607982.00,675278.00,-801455.00,-1035557.00,-977300.00,25.89,-30.73,-37.47,-12.51
EMP_A,2025-07,3825991.00,2929698.00,1049767.00,-583846.00,-677138.00,-895274.00,35.83,-19.93,-30.56,12.34
EMP_A,2025-08,3878403.00,2939104.00,879450.00,-721798.00,-952264.00,-1039018.00,29.92,-24.56,-35.35,0.32
EMP_A,2025-09,3772596.00,2759649.00,1002558.00,-569914.00,-676834.00,-733159.00,36.33,-20.65,-26.57,-6.11
EMP_A,2025-10,3665821.00,2819195.00,911455.00,-538969.00,-763282.00,-855850.00,32.33,-19.12,-30.36,2.16


### Visualizando resultados no Databricks

Depois de executar a consulta, use os recursos de visualização disponíveis no resultado SQL do seu workspace para explorar uma série temporal. Uma boa opção é usar `competencia` no eixo X e `receita_liquida`, `ebitda` ou `lucro_liquido` no eixo Y, filtrando por empresa. A interface e as opções de gráfico podem variar na Free Edition.

**Perguntas para discussão:** qual empresa apresenta maior receita líquida? Qual tem melhor margem EBITDA? O crescimento de receita coincide com crescimento de lucro?

## 7. Testes de consistência: qualidade não termina na Silver

Os testes abaixo avaliam três dimensões: **volume** (quantas linhas existem), **unicidade** (IDs duplicados na Silver) e **reconciliação** (fórmulas da DRE). O resultado ideal é zero duplicatas e zero inconsistências financeiras.

O número de linhas Bronze não precisa ser igual ao Silver se houver rejeições ou duplicatas. Para um pipeline de produção, testes também devem cobrir distribuição de valores, completude por mês, reconciliação com ERP e variações anormais.

In [0]:
%sql
SELECT
 (SELECT COUNT(*) FROM default.bronze_dre_lancamentos) AS bronze_registros,
 (SELECT COUNT(*) FROM default.silver_dre_lancamentos) AS silver_registros,
 (SELECT COUNT(*) FROM default.silver_dre_rejeitados) AS rejeitados,
 (SELECT COUNT(*) FROM (SELECT id_lancamento FROM default.silver_dre_lancamentos GROUP BY id_lancamento HAVING COUNT(*)>1)) AS duplicatas_silver,
 (SELECT COUNT(*) FROM default.gold_dre_mensal WHERE ABS(receita_liquida-(receita_bruta-deducoes))>0.01 OR ABS(lucro_liquido-(ebit+resultado_financeiro-ir_csll))>0.01) AS inconsistencias_dre;

bronze_registros,silver_registros,rejeitados,duplicatas_silver,inconsistencias_dre
60000,60000,0,0,0


## 8. Exercício prático: simule ingestão incremental

1. Retorne à célula de geração e altere `LOTE_ID = 1` para `LOTE_ID = 2`.
2. Execute novamente **a geração e o MERGE da Bronze**.
3. Reexecute a criação da Silver e as consultas Gold.
4. Compare a quantidade de linhas e os indicadores antes e depois.
5. Execute o mesmo `LOTE_ID = 2` outra vez: o total de IDs na Bronze não deve crescer.

**Desafio adicional:** crie uma consulta SQL que mostre o melhor mês por empresa segundo `margem_liquida_pct`, usando `ROW_NUMBER()`.

## 9. Glossário Databricks e Spark

| Termo | Explicação prática |
|---|---|
| Workspace | Ambiente de desenvolvimento com notebooks e objetos de dados |
| Notebook | Documento executável com código, resultados e explicações |
| Compute serverless | Recursos de processamento gerenciados pelo Databricks |
| Apache Spark | Motor distribuído para processar dados em escala |
| DataFrame | Estrutura tabular distribuída manipulada por APIs Spark |
| Spark SQL | Linguagem SQL executada pelo motor Spark |
| Delta Lake | Formato de tabela com transações e operações de atualização |
| `MERGE` | Insere ou atualiza linhas conforme uma chave |
| View | Consulta SQL reutilizável sem materializar os dados por padrão |
| Medallion | Organização em Bronze (bruto), Silver (confiável) e Gold (negócio) |

## 10. Encerramento e próximos passos

**Você construiu:** ingestão em Python → tabela Bronze Delta → saneamento Silver em SQL → DRE Gold em SQL → testes e análise temporal.

**Para evoluir em produção:** substituir o gerador por uma fonte real; aplicar regras de competência e estorno; usar dimensão de plano de contas; implementar trilha de auditoria e reconciliação; planejar atualizações incrementais, monitoramento, permissões e orquestração conforme os recursos disponíveis no seu plano.

**Limites deste laboratório:** micro-lotes manuais, ausência de streaming contínuo, reconstrução integral da Silver e dados financeiros fictícios. Não requer recursos exclusivos Enterprise.